In [1]:
!pip install pywin32



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import win32com.client

def run_sap2000():
    # مسیر نصب SAP2000 را وارد کنید
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"

    # بررسی وجود فایل اجرایی
    if not os.path.exists(program_path):
        print("SAP2000 executable not found at:", program_path)
        return

    # ساخت helper
    helper = win32com.client.Dispatch("SAP2000v1.Helper")

    # ساخت شی اصلی SAP2000
    sap_object = helper.CreateObject(program_path)

    # اجرای برنامه
    sap_object.ApplicationStart()

    # دسترسی به مدل
    sap_model = sap_object.SapModel

    # مقداردهی اولیه مدل جدید
    ret = sap_model.InitializeNewModel()

    # ایجاد یک قاب 2 بعدی (نمونه)
    ret = sap_model.File.New2DFrame("PortalFrame", 3, 124, 3, 200)

    # بستن SAP2000 در پایان (اختیاری)
    sap_object.ApplicationExit(False)

    # پاک‌سازی منابع
    sap_model = None
    sap_object = None

if __name__ == "__main__":
    run_sap2000()


ValueError: invalid literal for int() with base 10: 'PortalFrame'

In [1]:
import os
import win32com.client


def run_sap2000():
    # مسیر نصب SAP2000
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"

    # مسیر فایل مدل ذخیره شده
    model_path = r"C:\Users\ss\Desktop\automation\m2.sdb"  # ← آدرس فایل .sdb خودت رو جایگزین کن

    # بررسی وجود فایل اجرایی و مدل
    if not os.path.exists(program_path):
        print("SAP2000 executable not found at:", program_path)
        return

    if not os.path.exists(model_path):
        print("Model file not found at:", model_path)
        return

    # ساخت Helper و شیء SAP2000
    helper = win32com.client.Dispatch("SAP2000v1.Helper")
    sap_object = helper.CreateObject(program_path)
    sap_object.ApplicationStart()
    sap_model = sap_object.SapModel

    # باز کردن فایل مدل
    ret = sap_model.File.OpenFile(model_path)
    if ret != 0:
        print("خطا در باز کردن فایل مدل.")
        return

    # گرفتن شی طراحی فولاد
    steel_design = sap_model.DesignSteel

    # تعیین کد طراحی
    ret = steel_design.SetCode("AISC360-16")
    if ret != 0:
        print("خطا در تعیین کد طراحی فولاد.")
        return

    # اجرای طراحی فولاد
    ret = steel_design.StartDesign()
    if ret != 0:
        print("خطا در اجرای طراحی فولاد.")
        return

    # گرفتن نتایج طراحی
    Obj = []
    Elm = []
    Loc = []
    StepType = []
    StepNum = []
    Ratio = []
    Stage = []
    sError = []

    ret = steel_design.GetSummaryResults(Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError)

    # چاپ نتایج Design Ratio
    if ret == 0 and len(Ratio) > 0:
        print("نتایج طراحی اعضای سازه‌ای (Design Ratio):")
        for i in range(len(Ratio)):
            print(f"عضو {Obj[i]} - Ratio: {Ratio[i]}")
    else:
        print("نتیجه طراحی یافت نشد یا خطا وجود دارد.")

    # بستن SAP2000 (اگر خواستی بسته بشه)
    sap_object.ApplicationExit(False)

    # پاک کردن منابع
    sap_model = None
    sap_object = None


if __name__ == "__main__":
    run_sap2000()


خطا در تعیین کد طراحی فولاد.


In [3]:
import os
import win32com.client
import pythoncom
import pandas as pd
import subprocess

def run_sap2000():
    # مسیر نصب SAP2000
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"

    # مسیر فایل مدل ذخیره شده
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"  # ← آدرس فایل .sdb خودت رو جایگزین کن

    # مسیر برای ذخیره فایل اکسل
    excel_output_path = r"C:\Users\ss\Desktop\automation\design_results.xlsx"  # ← مسیر دلخواه برای فایل اکسل

    # بررسی وجود فایل اجرایی و مدل
    if not os.path.exists(program_path):
        print("SAP2000 executable not found at:", program_path)
        return

    if not os.path.exists(model_path):
        print("Model file not found at:", model_path)
        return

    try:
        # ساخت Helper و شیء SAP2000
        pythoncom.CoInitialize()  # Initialize COM
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        # باز کردن فایل مدل
        ret = sap_model.File.OpenFile(model_path)
        if ret != 0:
            print("خطا در باز کردن فایل مدل.")
            return

        # اجرای آنالیز مدل
        ret = sap_model.Analyze.RunAnalysis()
        if ret != 0:
            print("خطا در اجرای آنالیز.")
            return

        # گرفتن شی طراحی فولاد
        steel_design = sap_model.DesignSteel

        # تعیین کد طراحی
        ret = steel_design.SetCode("AISC360-16")
        if ret != 0:
            print("خطا در تعیین کد طراحی فولاد.")
            return

        # اجرای طراحی فولاد
        ret = steel_design.StartDesign()
        if ret != 0:
            print("خطا در اجرای طراحی فولاد.")
            return

        # گرفتن نتایج طراحی
        Obj = []
        Elm = []
        Loc = []
        StepType = []
        StepNum = []
        Ratio = []
        Stage = []
        sError = []

        ret = steel_design.GetSummaryResults(Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError)

        # نمایش نتایج Design Ratio (معادل گزینه Display)
        if ret == 0 and len(Ratio) > 0:
            print("نتایج طراحی اعضای سازه‌ای (Design Ratio):")
            for i in range(len(Ratio)):
                print(f"عضو {Obj[i]} - Ratio: {Ratio[i]}")
        else:
            print("نتیجه طراحی یافت نشد یا خطا وجود دارد.")

        # ایجاد جدول نتایج برای نمایش و ذخیره (معادل گزینه Show Table)
        results_data = {
            "Member": Obj,
            "Element": Elm,
            "Location": Loc,
            "Step Type": StepType,
            "Step Number": StepNum,
            "Design Ratio": Ratio,
            "Stage": Stage,
            "Error": sError
        }
        df = pd.DataFrame(results_data)

        # نمایش جدول در کنسول
        print("\nجدول نتایج طراحی:")
        print(df)

        # ذخیره جدول به صورت فایل اکسل
        df.to_excel(excel_output_path, index=False, sheet_name="Design Results")
        print(f"فایل اکسل در مسیر {excel_output_path} ذخیره شد.")

        # باز کردن فایل اکسل برای نمایش
        try:
            os.startfile(excel_output_path)  # برای ویندوز
        except Exception as e:
            print(f"خطا در باز کردن فایل اکسل: {e}")

        # بستن SAP2000
        sap_object.ApplicationExit(False)

    except Exception as e:
        print(f"خطا: {e}")
    finally:
        # پاک کردن منابع
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()  # Uninitialize COM

if __name__ == "__main__":
    run_sap2000()

خطا در تعیین کد طراحی فولاد.


In [4]:
import os
import win32com.client
import pythoncom
import pandas as pd

def export_steel_design_results():
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"
    excel_output_path = r"C:\Users\ss\Desktop\automation\design_results.xlsx"

    if not os.path.exists(program_path) or not os.path.exists(model_path):
        return  # خروج بی‌صدا اگر فایل‌ها موجود نیستند

    pythoncom.CoInitialize()
    try:
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        if sap_model.File.OpenFile(model_path) != 0:
            return

        sap_model.Analyze.RunAnalysis()

        steel_design = sap_model.DesignSteel
        steel_design.SetCode("AISC360-16")  # اگر این خطا داد، بررسی کن که کد موجود هست یا نه
        steel_design.StartDesign()

        Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError = [], [], [], [], [], [], [], []
        ret = steel_design.GetSummaryResults(Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError)

        if ret == 0 and len(Ratio) > 0:
            df = pd.DataFrame({
                "Member": Obj,
                "Element": Elm,
                "Location": Loc,
                "Step Type": StepType,
                "Step Number": StepNum,
                "Design Ratio": Ratio,
                "Stage": Stage,
                "Error": sError
            })
            df.to_excel(excel_output_path, index=False)

        sap_object.ApplicationExit(False)

    except Exception as e:
        pass  # خطاها را نادیده می‌گیریم چون فقط فایل خروجی مدنظر است
    finally:
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_steel_design_results()


In [5]:
import os
import win32com.client
import pythoncom
import pandas as pd
from win32com.client import VARIANT

def export_steel_design_results():
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"
    excel_output_path = r"C:\Users\ss\Desktop\automation\design_results.xlsx"

    if not os.path.exists(program_path) or not os.path.exists(model_path):
        print("فایل‌ها یافت نشدند.")
        return

    pythoncom.CoInitialize()
    try:
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        if sap_model.File.OpenFile(model_path) != 0:
            print("خطا در باز کردن فایل مدل")
            return

        sap_model.Analyze.RunAnalysis()

        steel_design = sap_model.DesignSteel
        steel_design.SetCode("AISC360-16")
        steel_design.StartDesign()

        # تعریف آرایه‌ها به صورت VARIANT برای COM
        Obj = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])
        Elm = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])
        Loc = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])
        StepType = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])
        StepNum = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_I4, [])
        Ratio = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_R8, [])
        Stage = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])
        sError = VARIANT(pythoncom.VT_ARRAY | pythoncom.VT_BSTR, [])

        ret = steel_design.GetSummaryResults(Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError)

        if ret == 0:
            # تبدیل VARIANT به لیست‌های پایتون
            df = pd.DataFrame({
                "Member": list(Obj.value),
                "Element": list(Elm.value),
                "Location": list(Loc.value),
                "Step Type": list(StepType.value),
                "Step Number": list(StepNum.value),
                "Design Ratio": list(Ratio.value),
                "Stage": list(Stage.value),
                "Error": list(sError.value)
            })
            df.to_excel(excel_output_path, index=False)
            print(f"فایل اکسل ذخیره شد در: {excel_output_path}")

            # باز کردن فایل اکسل
            try:
                os.startfile(excel_output_path)
            except Exception as e:
                print(f"خطا در باز کردن فایل اکسل: {e}")

        else:
            print("خطا در گرفتن نتایج طراحی")

        sap_object.ApplicationExit(False)

    except Exception as e:
        print("خطا:", e)

    finally:
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_steel_design_results()


خطا: int() argument must be a string, a bytes-like object or a real number, not 'VARIANT'


In [6]:
import os
import win32com.client
import pythoncom
import pandas as pd

def export_steel_design_results():
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"
    excel_output_path = r"C:\Users\ss\Desktop\automation\design_results.xlsx"

    if not os.path.exists(program_path) or not os.path.exists(model_path):
        print("فایل SAP2000 یا مدل یافت نشد.")
        return

    pythoncom.CoInitialize()
    try:
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        if sap_model.File.OpenFile(model_path) != 0:
            print("خطا در باز کردن فایل مدل.")
            return

        sap_model.Analyze.RunAnalysis()

        steel_design = sap_model.DesignSteel
        steel_design.SetCode("AISC360-16")
        steel_design.StartDesign()

        # تعریف لیست‌های خالی برای گرفتن خروجی از تابع COM
        Obj = []
        Elm = []
        Loc = []
        StepType = []
        StepNum = []
        Ratio = []
        Stage = []
        sError = []

        # گرفتن نتایج طراحی فولاد
        ret = steel_design.GetSummaryResults(Obj, Elm, Loc, StepType, StepNum, Ratio, Stage, sError)

        if ret == 0 and Ratio:
            df = pd.DataFrame({
                "Member": Obj,
                "Element": Elm,
                "Location": Loc,
                "Step Type": StepType,
                "Step Number": StepNum,
                "Design Ratio": Ratio,
                "Stage": Stage,
                "Error": sError
            })

            df.to_excel(excel_output_path, index=False)
            print(f"✅ فایل اکسل با موفقیت ذخیره شد:\n{excel_output_path}")

        else:
            print("⚠️ خطا در دریافت نتایج طراحی یا داده‌ای موجود نیست.")

        sap_object.ApplicationExit(False)

    except Exception as e:
        print(f"❌ خطا: {e}")

    finally:
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_steel_design_results()


❌ خطا: int() argument must be a string, a bytes-like object or a real number, not 'list'


In [7]:
import os
import win32com.client
import pythoncom
import pandas as pd

def export_steel_design_table():
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"
    excel_output_path = r"C:\Users\ss\Desktop\automation\design_results.xlsx"

    if not os.path.exists(program_path) or not os.path.exists(model_path):
        print("⛔ فایل SAP2000 یا مدل موجود نیست.")
        return

    pythoncom.CoInitialize()
    try:
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        # باز کردن مدل
        if sap_model.File.OpenFile(model_path) != 0:
            print("⛔ خطا در باز کردن فایل مدل.")
            return

        # اجرای آنالیز و طراحی
        sap_model.Analyze.RunAnalysis()
        steel_design = sap_model.DesignSteel
        steel_design.SetCode("AISC360-16")
        steel_design.StartDesign()

        # فعال‌سازی جدول‌های طراحی برای استخراج (مشابه Display > Show Tables)
        sap_model.DatabaseTables.SetLoadCasesSelectedForDisplay(["ALL"])
        sap_model.DatabaseTables.SetTablesSelectedForDisplay(["Steel Frame Design - Summary Data"])

        # استخراج داده‌ها از جدول‌ها
        table_keys, table_names, table_data = [], [], []

        ret = sap_model.DatabaseTables.GetAllTables(
            table_keys, table_names, table_data
        )

        if ret != 0 or not table_data:
            print("⛔ نتایج طراحی در جدول یافت نشد.")
            return

        # تبدیل جدول به DataFrame
        df_all = pd.DataFrame()

        for i in range(len(table_names)):
            table_name = table_names[i]
            if table_name == "Steel Frame Design - Summary Data":
                data = table_data[i]
                headers = data[0]
                rows = data[1:]
                df = pd.DataFrame(rows, columns=headers)
                df_all = df_all.append(df, ignore_index=True)

        if not df_all.empty:
            df_all.to_excel(excel_output_path, index=False)
            print(f"✅ فایل اکسل طراحی ذخیره شد:\n{excel_output_path}")
            os.startfile(excel_output_path)
        else:
            print("⚠️ جدول طراحی فولاد خالی است.")

        sap_object.ApplicationExit(False)

    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")

    finally:
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_steel_design_table()


❌ خطای غیرمنتظره: <unknown>.SetTablesSelectedForDisplay


In [8]:
import os
import win32com.client
import pythoncom
import pandas as pd

def export_steel_design_report():
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"
    model_path = r"C:\Users\ss\Desktop\automation\modelFixed.sdb"
    report_path = r"C:\Users\ss\Desktop\automation\steel_design_report.csv"

    if not os.path.exists(program_path) or not os.path.exists(model_path):
        print("⛔ فایل SAP2000 یا مدل موجود نیست.")
        return

    pythoncom.CoInitialize()
    try:
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        # باز کردن مدل
        if sap_model.File.OpenFile(model_path) != 0:
            print("⛔ خطا در باز کردن فایل مدل.")
            return

        # اجرای آنالیز و طراحی فولاد
        sap_model.Analyze.RunAnalysis()
        steel_design = sap_model.DesignSteel
        steel_design.SetCode("AISC360-16")
        steel_design.StartDesign()

        # تعریف مسیر فایل گزارش CSV
        if os.path.exists(report_path):
            os.remove(report_path)  # اگر قبلا هست پاک کن

        # استخراج گزارش طراحی فولاد به فایل CSV
        ret = sap_model.Report.Export(
            "Steel Frame Design - Summary Data",  # نام گزارش (ممکنه با نسخه شما فرق داشته باشه)
            report_path,
            0  # 0: Text file, 1: Excel file (ممکنه CSV هم قبول کنه)
        )

        if ret != 0:
            print("⛔ خطا در گرفتن گزارش طراحی فولاد.")
            return

        # خواندن گزارش CSV و ذخیره به اکسل با pandas
        df = pd.read_csv(report_path)
        excel_output_path = report_path.replace(".csv", ".xlsx")
        df.to_excel(excel_output_path, index=False)

        print(f"✅ فایل اکسل گزارش طراحی فولاد ذخیره شد:\n{excel_output_path}")
        os.startfile(excel_output_path)

        sap_object.ApplicationExit(False)

    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")

    finally:
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_steel_design_report()


❌ خطای غیرمنتظره: <unknown>.Report


In [9]:
import os
import win32com.client
import pythoncom
import pandas as pd

def export_sap2000_analysis_log():
    # مسیر نصب SAP2000
    program_path = r"C:\Program Files\Computers and Structures\SAP2000 24\SAP2000.exe"

    # مسیر فایل مدل
    model_path = r"C:\Users\ss\Desktop\automation\m2.sdb"  # ← مسیر فایل .sdb خود را جایگزین کنید

    # مسیر ذخیره فایل اکسل
    excel_output_path = r"C:\Users\ss\Desktop\automation\analysis_log.xlsx"  # ← مسیر دلخواه برای فایل اکسل

    # بررسی وجود فایل اجرایی و مدل
    if not os.path.exists(program_path):
        print("SAP2000 executable not found at:", program_path)
        return

    if not os.path.exists(model_path):
        print("Model file not found at:", model_path)
        return

    try:
        # Initialize COM
        pythoncom.CoInitialize()

        # ساخت Helper و شیء SAP2000
        helper = win32com.client.Dispatch("SAP2000v1.Helper")
        sap_object = helper.CreateObject(program_path)
        sap_object.ApplicationStart()
        sap_model = sap_object.SapModel

        # باز کردن فایل مدل
        ret = sap_model.File.OpenFile(model_path)
        if ret != 0:
            print("خطا در باز کردن فایل مدل.")
            return

        # اجرای آنالیز
        ret = sap_model.Analyze.RunAnalysis()
        if ret != 0:
            print("خطا در اجرای آنالیز.")
            return

        # گرفتن پیام‌های آنالیز (Analysis Log)
        LogText = []
        ret = sap_model.Analyze.GetRunLog(LogText)
        if ret != 0:
            print("خطا در دریافت پیام‌های آنالیز.")
            return

        # بررسی و نمایش پیام‌های آنالیز
        if LogText and len(LogText) > 0:
            print("پیام‌های آنالیز:")
            for message in LogText:
                print(message)

            # تبدیل پیام‌ها به DataFrame برای ذخیره در اکسل
            df = pd.DataFrame(LogText, columns=["Analysis Message"])

            # ذخیره در فایل اکسل
            df.to_excel(excel_output_path, index=False, sheet_name="Analysis Log")
            print(f"پیام‌های آنالیز در فایل اکسل ذخیره شد: {excel_output_path}")

            # باز کردن فایل اکسل (اختیاری)
            try:
                os.startfile(excel_output_path)  # برای ویندوز
            except Exception as e:
                print(f"خطا در باز کردن فایل اکسل: {e}")
        else:
            print("هیچ پیام آنالیزی یافت نشد.")

        # بستن SAP2000
        sap_object.ApplicationExit(False)

    except Exception as e:
        print(f"خطا: {e}")
    finally:
        # پاک کردن منابع
        sap_model = None
        sap_object = None
        pythoncom.CoUninitialize()

if __name__ == "__main__":
    export_sap2000_analysis_log()

خطا: <unknown>.GetRunLog
